# Reversible Fractional Attention — demo notebook

Cleaned, reproducible companion to the `rfa` package. Each section mirrors one of the scripts under `examples/`.

Repository: https://github.com/your-org/reversible-fractional-attention  
License: Apache-2.0

In [ ]:
import torch
from rfa import (
    ReversibleFractionalAttention,
    ParallelReversibleAttention,
    ComplexReversibleAttention,
    QuantizedStreamDelta,
    Rank1Stream,
)
torch.manual_seed(42)
print('torch', torch.__version__)

## 1. Forward pass with O(1) state

Run 100 tokens through the step-wise recurrence. The final accumulator shape is `[1, 64, 64]` regardless of sequence length.

In [ ]:
d_model, batch_size, seq_len = 64, 1, 100
model = ReversibleFractionalAttention(d_model=d_model, decay_rate=0.98)
state = model.initial_state(batch_size=batch_size)
inputs = torch.randn(seq_len, batch_size, d_model)
for x_t in inputs:
    step = model.step_forward(x_t, state)
    state = step.state
n_final = state[0]
print(f'Final N shape: {tuple(n_final.shape)}')
print(f'Final N bytes: {n_final.element_size() * n_final.nelement()} (independent of seq_len)')

## 2. Strict reversibility

Recover the state at `t-1` from the state at `t` and the streamed delta. Reconstruction error should be at float32 precision (~1e-6).

In [ ]:
state = model.initial_state(batch_size=batch_size)
states_history = [state]
deltas = []
for x_t in inputs:
    step = model.step_forward(x_t, state)
    state = step.state
    states_history.append(state)
    deltas.append(step.delta)
n_end, z_end = states_history[-1]
n_rec, z_rec = model.step_backward_reconstruct(n_end, z_end, deltas[-1])
n_true, z_true = states_history[-2]
print(f'max N error: {torch.max(torch.abs(n_rec - n_true)).item():.2e}')
print(f'max Z error: {torch.max(torch.abs(z_rec - z_true)).item():.2e}')

## 3. INT8 streaming stress test

Rewind 30 steps via an INT8-quantised delta stream. Accumulated relative error stays under ~2%.

In [ ]:
steps_back = 30
n_curr, z_curr = states_history[-1]
errors = []
for i in range(1, steps_back + 1):
    raw_dn, raw_dz = deltas[-i]
    q_dn, s_dn = QuantizedStreamDelta.compress(raw_dn)
    q_dz, s_dz = QuantizedStreamDelta.compress(raw_dz)
    dec_dn = QuantizedStreamDelta.decompress(q_dn, s_dn)
    dec_dz = QuantizedStreamDelta.decompress(q_dz, s_dz)
    n_curr, z_curr = model.step_backward_reconstruct(n_curr, z_curr, (dec_dn, dec_dz))
    true_n, _ = states_history[-1 - i]
    rel = (torch.norm(n_curr - true_n) / torch.norm(true_n)).item()
    errors.append(rel)
print(f'1 step:  {errors[0]*100:.2f}%')
print(f'10 steps: {errors[9]*100:.2f}%')
print(f'30 steps: {errors[-1]*100:.2f}%')

## 4. Rank-1 streaming (128x compression)

Stream only `(k, v)` instead of the full `d x d` matrix. Per-step footprint drops from 16 KB to 128 B for `d=64`.

In [ ]:
rank1_buffer = []
for x_t in inputs:
    k_t = model.kernel(model.w_k(x_t))
    v_t = model.w_v(x_t)
    rank1_buffer.append(Rank1Stream.pack(k_t, v_t))
fp32_bytes = d_model * d_model * 4
rank1_bytes = (d_model + d_model) * 1
print(f'FP32 matrix: {fp32_bytes} bytes')
print(f'Rank-1 INT8: {rank1_bytes} bytes ({fp32_bytes // rank1_bytes}x compression)')
n_curr, z_curr = states_history[-1]
errors = []
for i in range(1, steps_back + 1):
    rec_n, rec_z = Rank1Stream.unpack_and_reconstruct(*rank1_buffer[-i])
    n_curr, z_curr = model.step_backward_reconstruct(n_curr, z_curr, (rec_n, rec_z))
    true_n, _ = states_history[-1 - i]
    rel = (torch.norm(n_curr - true_n) / torch.norm(true_n)).item()
    errors.append(rel)
print(f'1 step:  {errors[0]*100:.2f}%')
print(f'30 steps: {errors[-1]*100:.2f}%')

## 5. Parallel training form

Vectorised form for training; mathematically equivalent to the step-wise form. Learn a tiny associative-recall task end-to-end.

In [ ]:
import torch.nn as nn
d_model, seq_len, batch_size, epochs = 32, 64, 16, 150
pmodel = ParallelReversibleAttention(d_model=d_model, max_len=seq_len)
optimizer = torch.optim.AdamW(pmodel.parameters(), lr=0.01)
target_projection = nn.Linear(d_model, d_model, bias=False)
for epoch in range(epochs + 1):
    optimizer.zero_grad()
    x = torch.randn(batch_size, seq_len, d_model)
    y_target = target_projection(x[:, 0, :])
    out = pmodel(x)
    loss = nn.functional.mse_loss(out[:, -1, :], y_target)
    loss.backward()
    optimizer.step()
    if epoch % 30 == 0:
        with torch.no_grad():
            gamma = pmodel.current_gamma().item()
        print(f'epoch {epoch:3d} | loss {loss.item():.6f} | gamma {gamma:.4f}')

## 6. Complex-valued selective attention

Per-channel learnable phase rotation makes the recurrence selective in *what* is stored, not only in *when*.

In [ ]:
cmodel = ComplexReversibleAttention(d_model=32, decay_rate=0.95)
phase = cmodel.phase_rotation()
print(f'phase magnitudes (all ~1): {phase.abs().tolist()[:8]}')
state = cmodel.initial_state(batch_size=1)
x = torch.randn(1, 32)
step = cmodel.step_forward(x, state)
n_rec, z_rec = cmodel.step_backward_reconstruct(*step.state, step.delta)
print(f'N reconstruction error: {torch.max(torch.abs(n_rec - state[0])).item():.2e}')